In [5]:
import torch
import torch.nn as nn
import numpy as np
import json
import pandas as pd
from pathlib import Path

In [6]:
DATA_ROOT = Path("movie_book_cdsr_processed/")
EMB_DIM = 512  # 文本/图像特征维度
with open(DATA_ROOT / "item2id.json", "r", encoding="utf-8") as f:
    item2id = json.load(f)
NUM_ITEMS = len(item2id)
print(f"全局物品总数: {NUM_ITEMS}")

全局物品总数: 132015


### 图像特征统一ID

In [12]:
# 加载图像特征与ID映射
image_emb = np.load(DATA_ROOT / "multimodal_features/image_features_512.npy")
image_id_map_df = pd.read_csv(DATA_ROOT / "multimodal_features/image_id_map.csv")
# 假设CSV列名: item_id, feature_index
image_id_map = {row['item_id']: idx for idx, row in image_id_map_df.iterrows()}
print(image_emb)

# 初始化图像嵌入层（可学习占位符）
image_embedding = nn.Embedding(NUM_ITEMS, EMB_DIM)
nn.init.uniform_(image_embedding.weight, a=-0.01, b=0.01)  # 与CLIP分布一致初始化
filled = 0
for item_id, feat_idx in image_id_map.items():
    # 直接使用 item_id 作为索引
    if 0 <= item_id < NUM_ITEMS:
        image_embedding.weight.data[item_id] = torch.from_numpy(image_emb[feat_idx])
        filled += 1
print(filled)

np.save(f"{DATA_ROOT}/multimodal_features_last/image_features_512.npy", image_embedding.weight.data.numpy())    

print(f"图像嵌入矩阵形状: {image_embedding.weight.shape}")
print(f"有效图像特征数: {len(image_id_map)}")

[[ 0.11229364  0.06593518  0.6986099  ... -0.06213899 -0.4710427
   0.00715558]
 [-0.7933111   0.4053588  -0.18113063 ...  0.31931865 -0.63353014
   0.3861462 ]
 [-0.21420553  0.14901966 -0.02839622 ...  0.8244412   0.12072357
   0.01802798]
 ...
 [ 0.24640623 -0.3026014   0.0914775  ...  0.20091085  0.15743217
   0.39904374]
 [ 0.02826975 -0.1394526   0.12492085 ...  0.11256284  0.5735604
  -0.29509786]
 [ 0.09874995  0.24704036 -0.23544423 ... -0.03011467 -0.18423544
  -0.15455297]]
95661
图像嵌入矩阵形状: torch.Size([132015, 512])
有效图像特征数: 95661


### CLIP统一ID

In [8]:
text_clip_emb = np.load(DATA_ROOT / "multimodal_features/text_features_clip_512.npy")
text_clip_id_map_df = pd.read_csv(DATA_ROOT / "multimodal_features/text_id_map_clip.csv")
text_clip_id_map = {row['item_id']: idx for idx, row in text_clip_id_map_df.iterrows()}
print(text_clip_emb)
# 初始化CLIP文本嵌入层
text_clip_embedding = nn.Embedding(NUM_ITEMS, EMB_DIM)
nn.init.uniform_(text_clip_embedding.weight, a=-0.01, b=0.01)

# 填充已有CLIP文本特征
for item_id, feat_idx in text_clip_id_map.items():
    # 直接使用 item_id 作为索引
    if 0 <= item_id < NUM_ITEMS:
        text_clip_embedding.weight.data[item_id] = torch.from_numpy(text_clip_emb[feat_idx])

np.save(f"{DATA_ROOT}/multimodal_features_last/text_features_clip_512.npy", text_clip_embedding.weight.data.numpy())

print(f"CLIP文本嵌入矩阵形状: {text_clip_embedding.weight.shape}")
print(f"有效CLIP文本特征数: {len(text_clip_id_map)}")

[[ 0.06866293  0.0939838   0.1320829  ...  0.18741585 -0.16292444
  -0.04632061]
 [-0.2667572   0.21182558  0.26856092 ... -0.05622216 -0.1601767
   0.07084402]
 [ 0.13023666  0.15333878 -0.25155026 ... -0.18325526 -0.04169858
  -0.17525762]
 ...
 [ 0.00123385 -0.06469532  0.15430574 ...  0.04798581  0.23384736
   0.20440485]
 [ 0.23144802 -0.03691545  0.20004992 ...  0.03364937  0.15625302
  -0.545955  ]
 [ 0.23632771  0.18158424  0.21202743 ... -0.12097144 -0.2985467
  -0.12048927]]
CLIP文本嵌入矩阵形状: torch.Size([132015, 512])
有效CLIP文本特征数: 95661


### Long-CLIP 统一ID

In [10]:
text_longclip_emb = np.load(DATA_ROOT / "multimodal_features/text_features_longclip_aligned_512.npy")
text_longclip_id_map_df = pd.read_csv(DATA_ROOT / "multimodal_features/text_id_map_longclip.csv")
text_longclip_id_map = {row['item_id']: idx for idx, row in text_longclip_id_map_df.iterrows()}
print(text_longclip_emb)
# 初始化Long-CLIP文本嵌入层
text_longclip_embedding = nn.Embedding(NUM_ITEMS, EMB_DIM)
nn.init.uniform_(text_longclip_embedding.weight, a=-0.01, b=0.01)

# 填充已有Long-CLIP文本特征
for item_id, feat_idx in text_longclip_id_map.items():
    if 0 <= item_id < NUM_ITEMS:
        text_longclip_embedding.weight.data[item_id] = torch.from_numpy(text_longclip_emb[feat_idx])

np.save(f"{DATA_ROOT}/multimodal_features_last/text_features_longclip_512.npy", text_longclip_embedding.weight.data.numpy())

print(f"Long-CLIP文本嵌入矩阵形状: {text_longclip_embedding.weight.shape}")
print(f"有效Long-CLIP文本特征数: {len(text_longclip_id_map)}")

[[-1.51302814e-01  1.29968715e+00 -1.10571158e+00 ... -2.67394930e-02
   2.43064255e-01  1.31935507e-01]
 [ 9.10618782e-01  2.36057281e+00  4.10985589e-01 ... -1.80535764e-02
  -1.94379225e-01 -1.93201303e-02]
 [ 8.06764603e-01  2.85492539e+00 -1.36484623e-01 ... -1.44864902e-01
  -2.03305483e-03 -3.79567087e-01]
 ...
 [ 4.22825336e+00 -1.35488617e+00  9.23730135e-02 ... -3.03241491e-01
  -8.59404951e-02  1.01249516e-01]
 [-3.10482550e+00 -1.07272577e+00  7.36207604e-01 ...  9.97535586e-02
   3.39956209e-02  8.76289606e-03]
 [ 1.31835413e+00  2.27282858e+00  1.32163298e+00 ... -6.32449538e-02
  -1.92323402e-01 -3.26345384e-01]]
Long-CLIP文本嵌入矩阵形状: torch.Size([132015, 512])
有效Long-CLIP文本特征数: 95661


In [13]:
import numpy as np

collab_emb = np.load("movie_book_cdsr_processed/multimodal_features/item_id_collab_512.npy")

print("形状:", collab_emb.shape)  # 应该是 (132012, 512)

total_zeros = np.count_nonzero(collab_emb == 0)
total_elements = collab_emb.size
print(f"总元素数: {total_elements}")
print(f"值为0的元素数: {total_zeros}")
print(f"非零元素数: {np.count_nonzero(collab_emb)}")

print("特征均值:", collab_emb.mean())
print("特征标准差:", collab_emb.std())
print(collab_emb)

形状: (132015, 512)
总元素数: 67591680
值为0的元素数: 2724864
非零元素数: 64866816
特征均值: 0.00068026356
特征标准差: 0.053784378
[[-0.03108074  0.0020148  -0.00168592 ...  0.05218666 -0.02823885
  -0.01343829]
 [-0.03865435  0.00642323 -0.00283191 ...  0.08010761 -0.04898849
  -0.03251971]
 [-0.02768935  0.01644679 -0.00850287 ...  0.0730785  -0.06473583
  -0.03013147]
 ...
 [-0.02205116  0.00194095  0.00388373 ...  0.03812687 -0.02270475
  -0.00633006]
 [-0.02113052 -0.00574212  0.00124321 ...  0.02899771 -0.0079597
  -0.00402978]
 [-0.02103809  0.00086453  0.00308022 ...  0.03826923 -0.0204573
  -0.00729826]]
